# Elevator simulation

A simulation of a multiple elevators operating efficiently in a tall busy building

In [57]:
# parameteric inputs
nFloors = 50
nPassengerIds = 100
capacityPerElevator = 8
nElevators = 2

# program params
input_data_fn = "input.csv"
elevator_position_history_fn = "out/elevator_position_history.csv"
summary_stats_fn = "out/summary_stats.csv"

nTimeSteps = 0 # will be set based on input data length


In [ ]:
# imports
import pandas as pd

# built-ins
from typing import Literal, TypedDict

In [64]:
# load input data
input_data = pd.read_csv(input_data_fn)

nTimeSteps = int(input_data.iloc[-1, 0])
print("number of time steps:", nTimeSteps)

# print(input_data)


number of time steps: 986


In [59]:
# validate input parameters
assert nFloors > 0, "Number of floors must be positive"
assert nPassengerIds > 0, "Number of passenger IDs must be positive"
assert capacityPerElevator > 0, "Capacity per elevator must be positive"
assert nElevators > 0, "Number of elevators must be positive"
assert nTimeSteps > 0, "Number of time steps must be positive"

In [60]:
# initialise the elevator position output data structure
elevator_position_history = pd.DataFrame({
    "time": range(nTimeSteps),
    **{f"elevator{i}": [0] * nTimeSteps for i in range(nElevators)}, # assumption: all elevators start on floor 0
}, dtype=int)

elevator_position_history.to_csv(elevator_position_history_fn, index=False)

In [ ]:
# a typesafe dict data structure for keeping track of active elevator requests
class ElevatorRequest(TypedDict):
    time: int
    source: int
    dest: int
    status: Literal["waiting", "travelling"]
    elevator: int | None

active_requests: dict[str, ElevatorRequest] = {}

In [ ]:
# loop through the input data and position the elevators efficiently
for t in range(nTimeSteps):
    # is there a request in this epoch?
    if t in input_data["time"].values:
        requests_at_t = input_data[input_data["time"] == t]
        for _, row in requests_at_t.iterrows():
            passenger_id = row["passenger_id"]
            active_requests[passenger_id] = {
                "time": row["time"],
                "source": row["source"],
                "dest": row["dest"],
                "status": "waiting",
                "elevator": None,
            }
    # update elevator positions for time t
    for elevator in range(nElevators):
        # simple strategy: keep elevators at floor 0 for now
        elevator_position_history.loc[t, f"elevator{elevator}"] = 0

elevator_position_history.to_csv(elevator_position_history_fn, index=False)
print(elevator_position_history)

     time  elevator0  elevator1
0       0          0          0
1       1          0          0
2       2          0          0
3       3          0          0
4       4          0          0
..    ...        ...        ...
981   981          0          0
982   982          0          0
983   983          0          0
984   984          0          0
985   985          0          0

[986 rows x 3 columns]
